# Генерация таблицы данных для M3 пайлайна

In [1]:
import sys
from dotenv import load_dotenv

sys.path.append('./../')
load_dotenv('./../.env')

True

In [2]:
import os
print(os.environ['OPENAI_API_KEY'])

YOUR_OPENAI_API_KEY


In [3]:
from typing import Any, List, TypedDict
import json
from langchain_openai import OpenAIEmbeddings
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
import numpy as np

from generators.m3_dag_generator.dag_generator import DagGenerationFlow, DagGenerationState
from generators.m3_dag_generator.md_generator import MDGenerationFlow, MDGenerationState

In [4]:
MODEL = 'gpt-oss-120b/latest' #'gpt-5.4-mini'
MODEL_PROVIDER = 'yandex'#'openai'
TEMPERATURE = 0.5

DAG_GEN_PROMPT = './../prompts/md_generation/m3_dag_gen/dag_generation.txt'
MD_GEN_PROMPT = './../prompts/md_generation/m3_dag_gen/md_generation.txt'
MAX_RETRIES = 20
RUNS = 8

SAVING_DIR = './../data/qwen/m3_dag_generations/'

def read_prompts(pr_path:str) -> str:
    with open(pr_path, 'rt') as fio:
        res = fio.read()
    return res 

In [5]:
import time
from dataclasses import dataclass
from typing import Optional
from langchain_core.callbacks import BaseCallbackHandler


class TokenCounter(BaseCallbackHandler):
    def __init__(self):
        self.prompt_tokens     = 0
        self.completion_tokens = 0
        self.total_tokens      = 0
        self.n_calls           = 0

    def on_llm_end(self, response, **kwargs):
        usage = None

        # LangChain wraps different providers differently
        if hasattr(response, "llm_output") and response.llm_output:
            usage = response.llm_output.get("token_usage") or \
                    response.llm_output.get("usage")

        # OpenAI via langchain_openai exposes it here
        if usage is None and response.generations:
            for gen_list in response.generations:
                for gen in gen_list:
                    info = getattr(gen, "generation_info", None) or {}
                    usage = info.get("usage") or info.get("token_usage")
                    if usage:
                        break

        if usage:
            self.prompt_tokens     += usage.get("prompt_tokens",     0)
            self.completion_tokens += usage.get("completion_tokens", 0)
            self.total_tokens      += usage.get("total_tokens",      0)
        self.n_calls += 1

In [6]:
class QueryDescr(TypedDict):
    id: str
    topic: str
    field_of_science: str
    llm_query: str

def read_tasks(tasks_file: str) -> List[str]:
    with open(tasks_file, 'rt') as fio:
        d = json.load(fio)
        queries_list = []
        for el in d:
            queries_list.append(QueryDescr(el))
    return queries_list

def generate_dag(md_gen_wf: DagGenerationFlow, query: str):
    counter = TokenCounter()

    md_gen_state = DagGenerationState()
    wf = md_gen_wf.workflow
    
    t0 = time.perf_counter()
    md_gen_state['users_task'] = query['llm_query']
    res_state = wf.invoke(
        md_gen_state,
        config={"callbacks": [counter]},
    )
    elapsed = time.perf_counter() - t0

    tokens_dict = {
        'prompt': counter.prompt_tokens,
        'completion': counter.completion_tokens,
        'total': counter.total_tokens
    }

    return res_state, elapsed, tokens_dict

def generate_meta(md_gen_wf: MDGenerationFlow, query: str, md_dag):
    counter = TokenCounter()

    md_gen_state = MDGenerationState()
    wf = md_gen_wf.workflow
    
    t0 = time.perf_counter()
    md_gen_state['users_task'] = query['llm_query']
    md_gen_state['md_name'] = md_dag['raw_dag'].md_name
    md_gen_state['md_description'] = md_dag['raw_dag'].md_descr
    md_gen_state['dag'] = md_dag['dag']
    
    res_state = wf.invoke(
        md_gen_state,
        config={
            "callbacks": [counter],
            "recursion_limit":100
            },
    )
    elapsed = time.perf_counter() - t0

    tokens_dict = {
        'prompt': counter.prompt_tokens,
        'completion': counter.completion_tokens,
        'total': counter.total_tokens
    }

    return res_state, elapsed, tokens_dict


In [ ]:
q_bank = read_tasks("./../queries/bio_queries.yaml")
dag_maker = DagGenerationFlow(
    read_prompts(DAG_GEN_PROMPT), 
    MODEL, 
    MODEL_PROVIDER, 
    TEMPERATURE,
    MAX_RETRIES
)
dag_state, el_t, tokens = generate_dag(dag_maker, q_bank[0])

In [ ]:
dag_state, el_t, tokens

In [ ]:
md_maker = MDGenerationFlow(
    read_prompts(MD_GEN_PROMPT), 
    MODEL, 
    MODEL_PROVIDER, 
    TEMPERATURE,
    5
)
md, el_t, tokens = generate_meta(md_maker, q_bank[0], dag_state)

In [ ]:
md, el_t, tokens

In [ ]:
total_fields = []
for node, fields in md['fields_per_node'].items():
    total_fields += fields.md_schema

In [ ]:
len(total_fields)

## Testing errors during per synthetic query

In [ ]:
import json
from dag_viz import visualize_dag

os.makedirs(SAVING_DIR, exist_ok=True)

q_bank = read_tasks("./../queries/bio_queries.yaml")
all_stats = []
for q_idx, q in enumerate(q_bank):
    for r_idx in range(RUNS):
        save_dir = os.path.join(SAVING_DIR, f'q_{q_idx}_r_{r_idx}')
        os.makedirs(save_dir, exist_ok=True)

        md_maker = DagGenerationFlow(
            read_prompts(DAG_GEN_PROMPT), 
            MODEL, 
            MODEL_PROVIDER, 
            TEMPERATURE,
            MAX_RETRIES
        )
        md, el_t, tokens = generate_meta(md_maker, q)
        print(f'DAG q={q_idx} r={r_idx} complete. Succes: {not md["generation_failed"]}. Tokens: {tokens["total"]}, iters: {md["rework_attempts"] + 1}.')
        
        stats = {
            'q': q_idx,
            'r': r_idx,
            'tokens_prompt': tokens['prompt'],
            'tokens_compl': tokens['completion'],
            'tokes_total': tokens['total'],
            'time': el_t,
            'success': not md["generation_failed"],
            'all_errors': md['all_dag_errors'],
            'reworks': md['rework_attempts']
        }
        with open(os.path.join(save_dir, 'stats.json'), 'wt') as fo:
            json.dump(stats, fo)
        all_stats.append(stats)

        if not md["generation_failed"]:
            visualize_dag(md['dag'], os.path.join(save_dir, 'schema_vis.html'), open_browser=False)
            with open(os.path.join(save_dir, 'dag.json'), 'wt') as fo:
                json.dump(md['raw_dag'].model_dump_json(), fo)


In [ ]:
import pandas as pd
df = pd.DataFrame(all_stats)
df.to_csv(os.path.join(SAVING_DIR, 'all_stats.csv'))

In [ ]:
df.head()

## 1. Генерация метаданных с DAG'ами

In [7]:
import pandas as pd
import os
import json
from dag_viz import visualize_dag


QUERIES_BANKS = {
    "synth_bio":'./../queries/bio_queries.yaml',
}

RUNS_CNT = 8
DEST_DICT = './../data/openai-oss/md_generations/m3'

dag_maker = DagGenerationFlow(
    read_prompts(DAG_GEN_PROMPT), 
    MODEL, 
    MODEL_PROVIDER, 
    TEMPERATURE,
    MAX_RETRIES
)

md_maker = MDGenerationFlow(
    read_prompts(MD_GEN_PROMPT), 
    MODEL, 
    MODEL_PROVIDER, 
    TEMPERATURE,
    MAX_RETRIES
)

dags_total_dir = os.path.join(DEST_DICT, 'm3_dags')
os.makedirs(os.path.join(DEST_DICT, 'm3_dags'), exist_ok=True)

bank_dicts = []
dags_stats = []
md_stats = []

In [8]:
# iterate over q_banks...
for q_name, q_path in QUERIES_BANKS.items():
    q_bank = read_tasks(q_path)
    
    # ...now over queries...
    for q_num, q in enumerate(q_bank):
        # ...and now over runs for averaging
        for run in range(RUNS_CNT):
            save_dir = os.path.join(dags_total_dir, f'q{q_num}_r{run}')
            os.makedirs(save_dir, exist_ok=True)
            
            # create DAG!
            is_complete = False
            while is_complete == False:
                dag_state, dag_el_t, dag_tokens = generate_dag(dag_maker, q)
                if dag_state['generation_failed'] == True:
                    print(f'DAG generation of q={q_num}, r={run} failed. Skipped.')
                else:
                    print(f'DAG q={q_num} r={run} complete. Tokens: {dag_tokens["total"]}, iters: {dag_state["rework_attempts"] + 1}.')
                    stats = {
                        'q': q_num,
                        'r': run,
                        'tokens_prompt': dag_tokens['prompt'],
                        'tokens_compl': dag_tokens['completion'],
                        'tokes_total': dag_tokens['total'],
                        'time': dag_el_t,
                        'success': not dag_state["generation_failed"],
                        'all_errors': dag_state['all_dag_errors'],
                        'reworks': dag_state['rework_attempts']
                    }
                    with open(os.path.join(save_dir, 'dag_stats.json'), 'wt') as fo:
                        json.dump(stats, fo)
                    dags_stats.append(stats)
                    visualize_dag(dag_state['dag'], os.path.join(save_dir, 'schema_vis.html'), open_browser=False)
                    with open(os.path.join(save_dir, 'dag.json'), 'wt') as fo:
                        json.dump(dag_state['raw_dag'].model_dump_json(), fo)
                    is_complete = True


            # create md!
            is_complete = False
            while is_complete == False:
                md_state, md_el_t, md_tokens = generate_meta(md_maker, q, dag_state)
                if md_state['md_gen_failed'] == True:
                    print(f'Failed to generate md on q={q_num}, r={run}. Skipping...')
                else:
                    md_stat = {
                        'q': q_num,
                        'r': run,
                        'tokens_prompt': md_tokens['prompt'],
                        'tokens_compl': md_tokens['completion'],
                        'tokes_total': md_tokens['total'],
                        'time': md_el_t,
                        'success': not md_state['md_gen_failed'],
                        'all_errors': md_state['total_issues'],
                        'reworks': md_state['total_reworks_cnt']
                    }
                    md_stats.append(md_stat)
                    with open(os.path.join(save_dir, 'md_stats.json'), 'wt') as fo:
                        json.dump(md_stat, fo)
                    is_complete = True
            

            # save md
            total_fields = 0
            md_name = dag_state['raw_dag'].md_name
            md_descr = dag_state['raw_dag'].md_descr
            for node_id, node in dag_state['dag'].nodes.items():
                for field in md_state['fields_per_node'][node_id].md_schema:
                    total_fields+=1
                    new_dict = {
                        'query_id': q_num,
                        'query_topic': q['topic'],
                        'query_field_of_science': q['field_of_science'],
                        'query_llm_query': q['llm_query'],
                        'run_id': run,
                        'md_short_name': md_name,
                        'md_description': md_descr,
                        'field_name': field.field_id,
                        'field_description': field.descr,
                        'field_type': field.type,
                        'node_id': node_id,
                        'node_name': node.name,
                        'node_descr': node.description,
                        'elapsed_time': dag_el_t + md_el_t,
                        'tokens_prompt': md_tokens['prompt'] + dag_tokens['prompt'],
                        'tokens_compl': md_tokens['completion'] + dag_tokens['completion'],
                        'tokes_total': md_tokens['total'] + dag_tokens['total'],
                    }
                    bank_dicts.append(new_dict)


            print(f'Finish processing: q={q_num}, "r_{run}"', 'elapsed_time', dag_el_t + md_el_t, 'tokens', md_tokens['completion'] + dag_tokens['completion'], 'fields:', total_fields)

    df = pd.DataFrame(bank_dicts)
    df.to_csv(os.path.join(DEST_DICT, f'm3_{q_name}_mds.csv'))

    dags_df = pd.DataFrame(dags_stats)
    dags_df.to_csv(os.path.join(DEST_DICT, f'm3_{q_name}_dag_stats.csv'))

    mds_df = pd.DataFrame(md_stats)
    mds_df.to_csv(os.path.join(DEST_DICT, f'm3_{q_name}_md_stats.csv'))


DAG q=0 r=0 complete. Tokens: 3396, iters: 1.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/openai-oss/md_generations/m3/m3_dags/q0_r0/schema_vis.html
Finish processing: q=0, "r_0" elapsed_time 777.256058662002 tokens 94480 fields: 486
DAG q=0 r=1 complete. Tokens: 3757, iters: 1.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/openai-oss/md_generations/m3/m3_dags/q0_r1/schema_vis.html
Finish processing: q=0, "r_1" elapsed_time 545.7652965690068 tokens 65757 fields: 401
DAG q=0 r=2 complete. Tokens: 4132, iters: 1.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/openai-oss/md_generations/m3/m3_dags/q0_r2/schema_vis.html
Finish processing: q=0, "r_2" elapsed_time 874.965896654001 tokens 101208 fields: 495
DAG q=0 r=3 complete. Tokens: 3964, iters: 1.
DAG saved → /home/alex/Documents/GitHub/mas_workflow/experiments/knn_evaluation/data/openai-oss/md_generations/m3/m3_dags/q0_r3

KeyboardInterrupt: 

In [ ]:
df = pd.DataFrame(bank_dicts)
df.to_csv(os.path.join(DEST_DICT, f'm3_{q_name}_mds.csv'), escapechar='\\')

dags_df = pd.DataFrame(dags_stats)
dags_df.to_csv(os.path.join(DEST_DICT, f'm3_{q_name}_dag_stats.csv'), escapechar='\\')

mds_df = pd.DataFrame(md_stats)
mds_df.to_csv(os.path.join(DEST_DICT, f'm3_{q_name}_md_stats.csv'), escapechar='\\')


## Compute embeds

In [ ]:
import os
import pandas as pd
from langchain_openai import OpenAIEmbeddings

QUERIES_CNT = 16
RUNS = 8

df = pd.read_csv('./../data/md_generations/m3_synth_bio_mds/m3_synth_bio_mds.csv', index_col=[0])
openai_embedder = OpenAIEmbeddings(model="text-embedding-3-large")


In [ ]:
from eval.md_classes import DataStep, MDField

In [ ]:
os.makedirs('./embeds/m3_fields_only/', exist_ok=True)

for idx, row in df.iterrows():
    #block = DataStep(row['node_id'], row['node_descr'])
    block = None
    md_field = MDField(row['field_name'], row['field_description'], row['field_type'], block=block)
    fields_str = str(md_field)

    emb = openai_embedder.embed_query(fields_str)
    emb = np.array(emb)
    np.save(os.path.join('./embeds/m3_fields_only/', f'{idx}.npy'), emb)
    print(f"(shape {emb.shape}, {idx}/{len(df)}) - '{md_field}' proceded.")